In [ ]:
# # 九曜：天墟 — 配音試作：BreezyVoice（台灣國語，Kaggle）第三版 2026-09-30
#
# Qwen3-TTS 設計的聲音偏粵語腔 → 改試 BreezyVoice（聯發科研究院，專為台灣國語訓練，Apache 2.0）。
# 拿 Qwen v3 挑出的 c2 參考聲音（旁白、蕭曜霖、机遙，已內嵌）讓 BreezyVoice 唸長篇 LF01 第 2、12 場的台詞，每句 3 次。
# 「九曜」的「曜」用注音 [:ㄧㄠ4] 指定讀音。
#
# Kaggle：右側 Settings → Accelerator 選 GPU T4 x2、Internet On → Save Version → Save & Run All（背景跑，關分頁沒關係）。
# 跑完：Output → /kaggle/working → EP01_voice_breezy_tiny.zip 下載傳給 Claude（失敗也照樣傳，裡面有 log_*.txt）。
#
# 第五版（2026-09-30）：旁白、蕭曜霖改用 OpenAI TTS 生成的台灣國語參考聲音（Qwen 參考帶粵語腔，BreezyVoice 會照學）。
# 第三版（2026-09-30 第 2 次執行：pip 建置 whisper 自動抓最新 setuptools 失敗 → 先單獨裝 whisper）改法：照 BreezyVoice 官方 Kaggle Playground（2026-09-27 仍可跑）的安裝方式。
#   ・環境是 Python 3.10（官方 Playground 同款舊環境）→ 直接裝在筆記本環境，最快。
#   ・環境是新版 Python → 自動另建 Python 3.10 環境（第二版做法，已修 whisper 建置、numpy／transformers 版本）。


In [ ]:
# 安裝；每一步的輸出都寫進 log_install.txt（會一起打包給 Claude 看）
import subprocess, os, sys, shutil, shlex
LOGDIR = "/kaggle/working/voice_breezy_tiny"; os.makedirs(LOGDIR, exist_ok=True)
def sh(cmd, log, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, env={**os.environ, "PYTHONUTF8": "1"})
    with open(f"{LOGDIR}/{log}", "a", encoding="utf-8") as f: f.write(f"\n$ {cmd}\n[exit {r.returncode}]\n{r.stdout[-20000:]}\n{r.stderr[-20000:]}\n")
    print(f"[exit {r.returncode}] {cmd[:110]}")
    if r.returncode: print("----- 錯誤訊息（最後 2500 字）-----\n" + (r.stdout[-1200:] + "\n" + r.stderr[-2500:]).strip() + "\n-----")
    return r.returncode
L = "log_install.txt"
sh("nvidia-smi --query-gpu=name,driver_version --format=csv; python -V", L)
shutil.rmtree("/tmp/BreezyVoice", ignore_errors=True)
sh("git clone -q https://github.com/mtkresearch/BreezyVoice.git /tmp/BreezyVoice", L)

if sys.version_info[:2] == (3, 10):
    # A：官方 Playground 做法（筆記本環境本身就是 3.10）；官方寫的 onnxruntime-gpu 1.20.1 已從 PyPI 下架 → 用 1.20.2；HyperPyYAML 1.2.2 遇新版 ruamel.yaml（0.18+）會報 max_depth 錯 → 鎖 <0.18
    PY = sys.executable; PIP = f"{PY} -m pip install"
    sh(f"{PY} -m pip uninstall -y onnxruntime", L)
    sh("(sudo apt-get -y install cudnn9-cuda-12 || apt-get -y install cudnn9-cuda-12)", L)
    open("/tmp/req_kaggle.txt", "w").write("\n".join([
        "conformer==0.3.2", "openai-whisper==20231117", "HyperPyYAML==1.2.2", "ruamel.yaml<0.18", "hydra-core==1.3.2",
        "lightning==2.2.4", "wget==3.2", "onnxruntime-gpu==1.20.2", "opencc-python-reimplemented", "g2pw"]))
    # openai-whisper 舊版打包要 pkg_resources；pip 建置時會抓最新版 setuptools（已移除 pkg_resources）→ 先裝舊版、關掉 build isolation 單獨裝
    sh(f"{PIP} 'setuptools<70' wheel", L)
    sh(f"{PIP} --no-build-isolation openai-whisper==20231117", L)
    sh(f"{PIP} -r /tmp/req_kaggle.txt", L)
    # g2pw 會順手裝回 CPU 版 onnxruntime，蓋掉 GPU 版 → 拿掉後重裝 GPU 版
    sh(f"{PY} -m pip uninstall -y onnxruntime; {PIP} --force-reinstall --no-deps onnxruntime-gpu==1.20.2", L)
    # 文字正規化：先試官方的 ttsfrd（modelscope 下載），失敗就用 WeTextProcessing
    if sh(f"{PIP} https://www.modelscope.cn/models/speech_tts/speech_kantts_ttsfrd/resolve/master/ttsfrd_dependency-0.1-py3-none-any.whl "
          "https://www.modelscope.cn/models/speech_tts/speech_kantts_ttsfrd/resolve/master/ttsfrd-0.3.9-cp310-cp310-linux_x86_64.whl", L) != 0:
        sh(f"{PIP} WeTextProcessing==1.0.3", L)
else:
    # B：新版 Kaggle 環境 → 另建 Python 3.10（放 /tmp，不進 Output）
    PY = "/tmp/bv/bin/python"
    sh("pip install -q uv", L); sh("uv python install 3.10", L); sh("uv venv -q -p 3.10 /tmp/bv", L)
    keep = [l.strip() for l in open("/tmp/BreezyVoice/requirements.txt") if l.strip() and not l.startswith("#")
            and not any(l.startswith(x) for x in ("gradio", "fastapi", "deepspeed", "tensorboard", "https://www.modelscope", "--"))]
    open("/tmp/req_kaggle.txt", "w").write("\n".join(keep))
    # 版本上限：torch 2.3 / onnxruntime-gpu 1.16 是 numpy 1.x 編的；g2pw 沒鎖 transformers，新版 5.x 不相容
    open("/tmp/constraints.txt", "w").write("numpy<2\ntransformers<4.50\ntorch==2.3.1\ntorchaudio==2.3.1\nruamel.yaml<0.18\n")
    PIP = ("uv pip install -p /tmp/bv/bin/python --index-strategy unsafe-best-match "
           "--extra-index-url https://download.pytorch.org/whl/cu118 -c /tmp/constraints.txt")
    # openai-whisper 舊版打包要 pkg_resources（新版 setuptools 已移除）→ 建置時限定舊 setuptools
    open("/tmp/build_constraints.txt", "w").write("setuptools<70\n")
    PIP += " -b /tmp/build_constraints.txt"
    if sh(f"{PIP} -r /tmp/req_kaggle.txt", L) != 0:
        for pkg in keep: sh(f"{PIP} {shlex.quote(pkg)}", L)  # 整包失敗就一個一個裝，記下哪個裝不起來
    # g2pw 會順手裝 CPU 版 onnxruntime，蓋掉 GPU 版 → 拿掉後重裝 GPU 版
    sh("uv pip uninstall -p /tmp/bv/bin/python onnxruntime", L)
    sh(f"{PIP} --reinstall --no-deps onnxruntime-gpu==1.16.0", L)

# 檢查：每個關鍵模組都要 import 得到，並記下 onnxruntime 有沒有 GPU
sh(f"{PY} -c \"import torch, torchaudio, whisper, onnxruntime, g2pw, opencc, numpy, transformers; "
   "print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'numpy', numpy.__version__, "
   "'transformers', transformers.__version__, 'ort', onnxruntime.__version__, onnxruntime.get_available_providers())\"", L)
sh(f"{PY} -c \"import ttsfrd; print('ttsfrd ok')\" || {PY} -c \"import tn; print('WeTextProcessing ok')\"", L)
print(open(f"{LOGDIR}/{L}", encoding="utf-8").read()[-1500:])


In [ ]:
# 參考聲音（v5：OpenAI gpt-4o-mini-tts 台灣國語男聲 ash／onyx，16kHz FLAC 內嵌，執行時轉回 WAV；Kaggle 筆記本上限 1MB）與台詞表
import base64, csv, json, os
REFS = json.loads(r'''{"VO_ash": {"text": "群山之上，雲海翻湧，一座古老的學院，靜靜矗立在那裡。晨光落在石階上，新的一年，又要開始了。", "b64": "..."}, "VO_onyx": {"text": "群山之上，雲海翻湧，一座古老的學院，靜靜矗立在那裡。晨光落在石階上，新的一年，又要開始了。", "b64": "..."}, "C05_ash": {"text": "所有人聽好，明天一早到演武場集合，誰都不准遲到。規矩只說一次，記清楚了。", "b64": "..."}, "C05_onyx": {"text": "所有人聽好，明天一早到演武場集合，誰都不准遲到。規矩只說一次，記清楚了。", "b64": "..."}}''')
ROWS = [["VO_ash", "LF01-A01_旁白_ash", "九曜[:ㄧㄠ4]界，天玄院。"], ["C05_ash", "LF01-A10a_蕭曜霖_ash", "不用等明天。"], ["C05_ash", "LF01-A10b_蕭曜霖_ash", "半刻鐘後，新生演武場集合。"], ["VO_onyx", "LF01-A01_旁白_onyx", "九曜[:ㄧㄠ4]界，天玄院。"], ["C05_onyx", "LF01-A10a_蕭曜霖_onyx", "不用等明天。"], ["C05_onyx", "LF01-A10b_蕭曜霖_onyx", "半刻鐘後，新生演武場集合。"]]
TAKES = 2
os.makedirs("/tmp/bv_ref", exist_ok=True); OUT = "/kaggle/working/voice_breezy"; os.makedirs(OUT, exist_ok=True)
for k, v in REFS.items():
    open(f"/tmp/bv_ref/{k}.flac", "wb").write(base64.b64decode(v["b64"]))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", f"/tmp/bv_ref/{k}.flac", "-ac", "1", "-ar", "16000", f"/tmp/bv_ref/{k}.wav"], check=True)
    os.remove(f"/tmp/bv_ref/{k}.flac")
with open("/tmp/bv_input.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["speaker_prompt_audio_filename", "speaker_prompt_text_transcription", "content_to_synthesize", "output_audio_filename"])
    for ref, name, text in ROWS:
        for t in range(1, TAKES + 1): w.writerow([ref, REFS[ref]["text"], text, f"{name}_t{t}"])
print(open("/tmp/bv_input.csv", encoding="utf-8").read())


In [ ]:
# 生成（第一次會自動下載模型）；輸出寫進 log_run.txt
sh(f"{PY} batch_inference.py --csv_file /tmp/bv_input.csv --speaker_prompt_audio_folder /tmp/bv_ref --output_audio_folder {OUT}", "log_run.txt", cwd="/tmp/BreezyVoice")
sh(f"find {OUT} -type f | sort", "log_run.txt")
print(open(f"{LOGDIR}/log_run.txt", encoding="utf-8").read()[-3000:])


In [ ]:
# 打包上傳用小檔（MP3＋安裝／執行紀錄）；就算失敗也會打包紀錄
import glob
for w in sorted(glob.glob(f"{OUT}/*.wav")):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-t", "15", "-ac", "1", "-b:a", "48k", f"{LOGDIR}/{os.path.basename(w)[:-4]}.mp3"])
shutil.make_archive("/kaggle/working/EP01_voice_breezy_tiny", "zip", LOGDIR)
print(len(glob.glob(f"{LOGDIR}/*.mp3")), "個聲音檔 →", f"EP01_voice_breezy_tiny.zip {os.path.getsize('/kaggle/working/EP01_voice_breezy_tiny.zip')/1e6:.1f} MB（含 log_*.txt）")

# Claude 從雲端抓不到 Output 檔（下載網址被擋）→ 把 MP3 以 base64 印進執行紀錄，由 Claude 從紀錄還原
import base64
for m in sorted(glob.glob(f"{LOGDIR}/*.mp3")):
    print(f"@@MP3 {os.path.basename(m)} {base64.b64encode(open(m, 'rb').read()).decode()}")
